# display version of python used 

In [1]:
!py --version

Python 3.10.8


# import the libary

In [2]:
from paddleocr import PaddleOCR 
import paddle
import cv2
import numpy as np
import os

D:\Anuj_S_Jain\Data\Code\Project\Society_MCE_QT\Society_MCE\Prototype\.venv\lib\site-packages\requests\__init__.py:113: RequestsDependencyWarning: urllib3 (2.6.3) or chardet (7.3.0)/charset_normalizer (3.4.6) doesn't match a supported version!
  warnings.warn(
D:\Anuj_S_Jain\Data\Code\Project\Society_MCE_QT\Society_MCE\Prototype\.venv\lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
Checking connectivity to the model hosters, this may take a while. To bypass this check, set `DISABLE_MODEL_SOURCE_CHECK` to `True`.
D:\Anuj_S_Jain\Data\Code\Project\Society_MCE_QT\Society_MCE\Prototype\.venv\lib\site-packages\paddle\utils\cpp_extension\extension_utils.py:718: UserWarning: No ccache found. Please be aware that recompiling all source files may be required. You can download and install ccache from: https://github.com/ccac

# device used

In [27]:
paddle.device.get_device()

'cpu'

# loading image path

In [71]:
i=1
image_path = f"D:/Anuj_S_Jain/Data/Code/Project/Society_MCE_QT/Society_MCE/Prototype/training data/diff_img/{i}.jpg"
image_path

'D:/Anuj_S_Jain/Data/Code/Project/Society_MCE_QT/Society_MCE/Prototype/training data/diff_img/1.jpg'

# read img

In [72]:
img = cv2.imread(image_path)

# run paddleocr on img 

In [18]:
if (img is None) :
    print(f"{image_path} this is incorrect no img found")
else:
    # setting tread 
    os.environ["OMP_NUM_THREADS"] = "8"
    os.environ["MKL_NUM_THREADS"] = "8"
    # applying ocr on the image
    ocr = PaddleOCR(use_doc_orientation_classify=False, 
        use_doc_unwarping=False, 
        use_textline_orientation=False,
        lang='en',
        device=paddle.device.get_device(),
        cpu_threads=8 
       )
    result = ocr.predict(img)

Creating model: ('PP-OCRv5_server_det', None)
Model files already exist. Using cached files. To redownload, please delete the directory manually: `C:\Users\Anuj S Jain\.paddlex\official_models\PP-OCRv5_server_det`.
Creating model: ('en_PP-OCRv5_mobile_rec', None)
Model files already exist. Using cached files. To redownload, please delete the directory manually: `C:\Users\Anuj S Jain\.paddlex\official_models\en_PP-OCRv5_mobile_rec`.


# make an array by combining the ploy points , text , and confident scores

In [20]:
ocr_result = [(poly , text , scores)
    for poly , text , scores  in 
        zip(result[0]['dt_polys'],
            result[0]['rec_texts'],
            result[0]['rec_scores'])]

# extracting the table from the img 

In [73]:
if (img is None) :
    print(f"{image_path} this is incorrect no img found")
else:
    # converting img to gray
    gray = cv2.cvtColor(img, cv2.COLOR_BGR2GRAY)
    # cv2.imwrite(f'./output/opencv/gray{i}.jpeg', gray)

    # splits the image and clahe 
    # C – Contrast
    # L – Limited
    # A – Adaptive
    # H – Histogram
    # E – Equalization
    clahe = cv2.createCLAHE(clipLimit=2.0, tileGridSize=(8,8))
    gray_clahe = clahe.apply(gray)    
    # cv2.imwrite(f'./output/opencv/gray_clahe{i}.jpeg', gray_clahe)
    
    # Better threshold (adaptive handles uneven lighting)
    thresh = cv2.adaptiveThreshold(
        gray_clahe, 255,
        cv2.ADAPTIVE_THRESH_MEAN_C,
        cv2.THRESH_BINARY_INV,
        15, 10
    )
    
    
    
    kernel = np.zeros((2,2), np.uint8)
    thresh = cv2.erode(thresh, kernel, iterations=2)
    cv2.imwrite(f'./output/opencv/thresh{i}.jpeg', thresh)
    
    
    # --- Horizontal lines ---
    horizontal_kernel = cv2.getStructuringElement(cv2.MORPH_RECT, (50,1))
    horizontal = cv2.morphologyEx(thresh, cv2.MORPH_OPEN, horizontal_kernel)
    # cv2.imwrite(f'./output/opencv/horizontal{i}.jpeg', horizontal)
    
    # connect broken horizontal lines
    horizontal = cv2.dilate(horizontal, np.ones((2,2),np.uint8), iterations=2)
    # cv2.imwrite(f'./output/opencv/horizontal_dilated{i}.jpeg', horizontal)
    
    # --- Vertical lines ---
    vertical_kernel = cv2.getStructuringElement(cv2.MORPH_RECT, (1,50))
    vertical = cv2.morphologyEx(thresh, cv2.MORPH_OPEN, vertical_kernel)
    # cv2.imwrite(f'./output/opencv/vertical{i}.jpeg', vertical)
     
    # connect broken vertical lines
    vertical = cv2.dilate(vertical, np.ones((2,2),np.uint8), iterations=2)
    # cv2.imwrite(f'./output/opencv/vertical_dilate{i}.jpeg', vertical)
    
    
    # Combine
    boxes = cv2.add(horizontal, vertical)
    cv2.imwrite(f'./output/opencv/boxes_combine{i}.jpeg', boxes)
    
    # Final closing to join gaps
    kernel = np.ones((10,10), np.uint8)
    boxes = cv2.morphologyEx(boxes, cv2.MORPH_CLOSE, kernel, iterations=2)
    cv2.imwrite(f'./output/opencv/boxes_fill_gap{i}.jpeg', boxes)   
    
    # Find contours
    contours, _ = cv2.findContours(boxes, cv2.RETR_TREE, cv2.CHAIN_APPROX_SIMPLE)
    # cv2.imwrite(f'./output/opencv/contours{i}.jpeg', contours)
    

# find and draw the counter

In [75]:
output = img.copy()

# for index,cnt in enumerate(contours):
#     x,y,w,h = cv2.boundingRect(cnt)
#     # filter noise
#     if w > 100 and h > 100:
cv2.drawContours(output, contours, -1, (0,255,0), 1)
cv2.imwrite(f'./output/opencv/output{i}.jpeg', output)

True